In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Penggunaan workload

<span id="usage"></span>
Usage adalah ukuran jumlah waktu QPU dikunci untuk workload kamu, dan dihitung secara berbeda tergantung pada mode eksekusi yang digunakan.

* Usage session adalah wall-clock time selama session aktif. Lihat [Panjang session](/guides/run-jobs-session#session-length) untuk informasi lebih lanjut tentang transisi status session.
* Usage batch adalah jumlah quantum time (waktu yang dihabiskan oleh kompleks QPU untuk memproses job kamu) dari semua job dalam batch.
* Usage single job adalah quantum time yang digunakan job dalam pemrosesan.

Perlu dicatat bahwa job yang gagal atau dibatalkan tetap dihitung dalam usage kamu dalam kondisi tertentu — lihat bagian [Job yang gagal dan dibatalkan](#failed-job) untuk detailnya.

Untuk pengguna paket berbayar, usage menentukan berapa biaya workload. Lihat [Kelola biaya](/guides/manage-cost) untuk detailnya.

<span id="failed-job"></span>
## Usage untuk job yang gagal dan dibatalkan
Ketika sebuah job gagal atau dibatalkan, usage yang dilaporkan adalah sebagai berikut:

* Mode job atau batch: Usage yang dilaporkan adalah waktu QPU dikunci untuk mengeksekusi workload kamu hingga saat gagal atau dibatalkan. Oleh karena itu, jika kegagalan atau pembatalan terjadi sebelum penguncian, usage yang dilaporkan adalah nol. Jika tidak, usage yang dilaporkan adalah jumlah usage sebelum workload gagal atau dibatalkan. Dengan demikian, beberapa job yang gagal tidak muncul dalam usage yang dilaporkan dan beberapa lainnya muncul.

* Mode session: Usage yang dilaporkan adalah wall-clock time selama session aktif, terlepas dari berapa banyak job yang gagal atau dibatalkan.

<span id="view-usage"></span>
## Melihat usage aktual sebuah workload
Setelah workload selesai, ada beberapa cara untuk melihat usage aktualnya:

- Jalankan [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) atau [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) di `qiskit-ibm-runtime` versi 0.30 atau lebih baru. Jika menggunakan versi `qiskit-ibm-runtime` yang lebih lama (>= 0.23 dan < 0.30), usage masih bisa ditemukan di `session.details()["usage_time"]` dan `batch.details()["usage_time"]`.
- Gunakan [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) untuk melihat usage untuk batch atau session tertentu.
- Gunakan [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) untuk melihat usage untuk satu job.

<span id="instance-usage"></span>
## Lihat usage instance
Kamu bisa melihat usage sebuah instance di halaman [Instances](https://quantum.cloud.ibm.com/instances), atau, bagi yang memiliki wewenang yang sesuai, halaman [Analytics](https://quantum.cloud.ibm.com/analytics). Perlu diperhatikan bahwa kedua halaman mungkin menampilkan angka usage yang berbeda karena cara penghitungannya berbeda.

Halaman Instances menampilkan usage real-time untuk 28 hari terakhir (bergulir), hingga waktu saat ini pada hari ini. Usage halaman Analytics dihitung ulang setiap jam dan mencakup 28 hari penuh terakhir; yaitu, menampilkan usage dari 00:00 28 hari lalu hingga hari ini, pada awal jam.

## Perkirakan usage sebelum mengirimkan job
Meskipun mendapatkan estimasi lokal yang akurat dipersulit oleh operasi tambahan yang dilakukan untuk error suppression dan mitigation, kamu bisa menggunakan rumus dasar ini untuk mendapatkan perkiraan estimasi usage:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` adalah overhead sekitar 2 detik per sub-job. Ini mencakup operasi seperti memuat payload ke dalam elektronik kontrol. Job primitive kamu mungkin dibagi menjadi beberapa sub-job jika terlalu besar untuk diproses sekaligus oleh execution engine.
- `rep_delay` adalah opsi yang [bisa dikustomisasi oleh pengguna](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), dan defaultnya diberikan oleh `backend.default_rep_delay`, yaitu 250 mikrodetik pada sebagian besar Backend IBM Quantum. Perlu diperhatikan bahwa menurunkan `rep_delay` mengurangi total waktu eksekusi QPU, tetapi dengan risiko meningkatnya error rate persiapan state; lihat panduan [Dynamic repetition rate execution](/guides/repetition-rate-execution) untuk informasi lebih lanjut.
- `<circuit length>` adalah total panjang instruksi. Setiap instruksi membutuhkan waktu yang berbeda di QPU, sehingga total panjangnya bervariasi dari Circuit ke Circuit. Pengukuran, misalnya, bisa memakan waktu 56 kali lebih lama daripada Gate `x`. `backend.target[<instruction>][<qubit>].duration` bisa digunakan untuk menemukan durasi tepat setiap instruksi. Panjang Circuit yang umum kemungkinan antara 50-100 mikrodetik. Jika kamu menggunakan teknik error suppression atau mitigation dengan primitive, instruksi tambahan mungkin disisipkan ke dalam Circuit kamu, yang akan meningkatkan total panjang Circuit.
    > **Note:** [Opsi eksperimental `scheduler_timing`](/guides/visualize-circuit-timing) mengembalikan total waktu Circuit, tetapi ini BUKAN waktu yang digunakan untuk penagihan.
- `<num executions>` adalah total jumlah Circuit dikalikan jumlah shot, di mana Circuit adalah yang dihasilkan setelah elemen PUB disiarkan.
    - Jika kamu menggunakan teknik error mitigation dengan primitive, Circuit tambahan mungkin dijalankan sebagai bagian dari proses mitigation, yang akan meningkatkan total jumlah eksekusi. Selain itu, teknik error mitigation lanjutan seperti PEA dan PEC memiliki overhead yang jauh lebih tinggi karena memerlukan menjalankan Circuit untuk noise learning.
    - Estimator mengelompokkan observable yang commuting secara qubit-wise, yang mengurangi jumlah eksekusi.

Jika kamu tidak menggunakan teknik error mitigation lanjutan atau `rep_delay` kustom, kamu bisa menggunakan `2+0.00035*<num executions>` sebagai rumus cepat.

## Langkah selanjutnya
> **Tip:** - Tinjau tips berikut: [Minimalkan waktu berjalan job](minimize-time).
>     - Atur [Waktu eksekusi maksimum](max-execution-time).
>     - Pelajari cara transpile secara lokal di bagian [Transpile](./transpile/).
>     - Coba panduan [Bandingkan pengaturan Transpiler](/guides/circuit-transpilation-settings).